# maimaiDX 脱敏成绩数据集 EDA

探索匿名玩家 B50 / 全量成绩 / Rating 趋势 / 同段 ARPI（`arpi_peer_stats`）。

- 数据已脱敏：无 QQ / 昵称，仅有单向哈希 `player_id`
- 适用于同段统计、锐评样本与推分可行性分析

## 关于 Δrating=337

相邻快照单次同步的「正常」变化上限约 **337**。  
更大跳变（如一次 +2000/+4000）通常不是脏数据，而是**累计游玩后统一上传/补存**——应保留，并单独标记为 `bulk_upload`，用 `ceil(|Δ|/337)` 估最少等效同步次数。
- 额外指标：B50 FC/AP/SSS+、定数地板、新曲压力、推分速度、全量 Master+ 渗透、拖后腿谱


In [ ]:
import json
import math
import os
import sys
import shutil
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.unicode_minus"] = False

# 相邻快照「单次正常同步」上限；超出记为 bulk（累计上传），不默认丢弃
MAX_STEP_DELTA = 337
RATING_MIN, RATING_MAX = 1, 17000


def _apply_font_stack(primary: str) -> str:
    """CJK 优先，DejaVu 兜底数学符号（如 ℝ），避免缺字形警告。"""
    stack = [primary, "DejaVu Sans", "Arial Unicode MS", "sans-serif"]
    # 去重保序
    seen, clean = set(), []
    for name in stack:
        if name and name not in seen:
            seen.add(name)
            clean.append(name)
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = clean
    return primary


def setup_plot_font() -> str:
    """启用 CJK 字体，避免日文/中文曲名显示为方框。"""
    prefer = [
        "Noto Sans CJK JP",
        "Noto Sans CJK SC",
        "Noto Serif CJK JP",
        "Source Han Sans JP",
        "Source Han Sans",
        "WenQuanYi Zen Hei",
        "WenQuanYi Micro Hei",
        "IPAGothic",
        "TakaoPGothic",
        "Yu Gothic",
        "Hiragino Sans",
        "Microsoft YaHei",
        "SimHei",
        "Arial Unicode MS",
    ]

    def _pick():
        available = {f.name for f in fm.fontManager.ttflist}
        for name in prefer:
            if name in available:
                return name
        for f in fm.fontManager.ttflist:
            low = (f.name + " " + f.fname).lower()
            if "noto" in low and "cjk" in low:
                return f.name
        return None

    hit = _pick()
    if hit:
        print("plot font:", _apply_font_stack(hit), "+ DejaVu fallback")
        return hit

    if shutil.which("apt-get"):
        try:
            subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(
                ["apt-get", "-qq", "-y", "install", "fonts-noto-cjk"],
                check=False,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            fm._load_fontmanager(try_read_cache=False)
            hit = _pick()
            if hit:
                print("plot font (apt):", _apply_font_stack(hit), "+ DejaVu fallback")
                return hit
        except Exception as e:
            print("apt font install skipped:", type(e).__name__, e)

    font_path = Path("/tmp/NotoSansCJKjp-Regular.otf")
    if not font_path.exists() or font_path.stat().st_size < 1000:
        url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/Japanese/NotoSansCJKjp-Regular.otf"
        try:
            print("downloading CJK font (may take a while)...")
            urllib.request.urlretrieve(url, font_path)
        except Exception as e:
            print("font download failed:", type(e).__name__, e)

    if font_path.exists() and font_path.stat().st_size > 1000:
        try:
            fm.fontManager.addfont(str(font_path))
            name = fm.FontProperties(fname=str(font_path)).get_name()
            print("plot font (file):", _apply_font_stack(name), "+ DejaVu fallback")
            return name
        except Exception as e:
            print("addfont failed:", type(e).__name__, e)

    _apply_font_stack("DejaVu Sans")
    print("plot font: DejaVu Sans (will use song_id labels for CJK titles)")
    return "DejaVu Sans"


def _has_cjk(text: str) -> bool:
    for ch in str(text or ""):
        o = ord(ch)
        if (
            0x3040 <= o <= 0x30FF  # JP kana
            or 0x3400 <= o <= 0x4DBF
            or 0x4E00 <= o <= 0x9FFF  # CJK
            or 0xF900 <= o <= 0xFAFF
            or 0xFF66 <= o <= 0xFF9D
        ):
            return True
    return False


def _sanitize_plot_text(text: str) -> str:
    """把 CJK 字体常缺的符号换成 ASCII，减少 Glyph missing 警告。"""
    import unicodedata

    s = str(text)
    # 数学花体/双线体字母（如 ℝ𝔼）→ 尽量归一成普通拉丁字母
    out = []
    for ch in s:
        o = ord(ch)
        if ch in "ℝℜℛ":
            out.append("R")
            continue
        if 0x1D400 <= o <= 0x1D7FF or 0x2100 <= o <= 0x214F:
            try:
                name = unicodedata.name(ch, "")
                # e.g. MATHEMATICAL DOUBLE-STRUCK CAPITAL E
                for token in name.split():
                    if len(token) == 1 and token.isalpha():
                        out.append(token)
                        break
                else:
                    decomp = unicodedata.normalize("NFKD", ch)
                    ascii_ch = "".join(c for c in decomp if c.isascii() and c.isalpha())
                    out.append(ascii_ch or "?")
            except Exception:
                out.append("?")
            continue
        out.append(ch)
    s = "".join(out)
    return (
        s.replace("Δ", "d")
        .replace("Σ", "sum")
        .replace("≠", "!=")
        .replace("≥", ">=")
        .replace("≤", "<=")
        .replace("×", "x")
        .replace("…", "...")
    )


def chart_axis_label(title, diff, song_id, font_name: str = "", max_len: int = 30) -> str:
    """有 CJK 字体用曲名；否则对非 ASCII 曲名回退 song_id，避免方框。"""
    t = _sanitize_plot_text(str(title or "").strip().replace("\n", " "))
    d = _sanitize_plot_text(str(diff or ""))
    sid = song_id if song_id is not None else "?"
    font_ok = font_name not in ("", "DejaVu Sans")
    if t and (font_ok or not _has_cjk(t)):
        if len(t) > max_len:
            t = t[: max_len - 1] + "..."
        return f"{t} [{d}]"
    return f"#{sid} [{d}]" + (f" {t[:18]}" if t and not _has_cjk(t) else "")


PLOT_FONT = setup_plot_font()

# 原始 rate 常见: sssp / SSSp / sss（不是 "SSS+" 字符串）
# 统一映射到展示态，避免再出现「全员 0 张 SSS+」这种低级错误
_RATE_MAP = {
    "sssp": "SSS+", "sss+": "SSS+",
    "sss": "SSS",
    "ssp": "SS+", "ss+": "SS+",
    "ss": "SS",
    "sp": "S+", "s+": "S+",
    "s": "S",
    "aaa": "AAA", "aa": "AA", "a": "A",
    "bbb": "BBB", "bb": "BB", "b": "B",
    "c": "C", "d": "D",
}


def norm_rate(rate) -> str:
    raw = str(rate or "").strip().replace("＋", "+")
    if not raw:
        return ""
    s = raw.lower()
    if s in _RATE_MAP:
        return _RATE_MAP[s]
    # 兼容已是 SSS+ / SS+ 的展示态
    up = raw.upper()
    if up in set(_RATE_MAP.values()):
        return up
    return up


def T(zh: str, en: str) -> str:
    """图标题中英对照。"""
    return f"{zh} / {en}"


# 相关矩阵等用的短标签（中英）
FEATURE_LABELS = {
    "rating": "Rating",
    "arpi": "ARPI",
    "record_count": "谱面数/records",
    "rating_delta": "窗口ΔR/delta",
    "trend_points": "趋势点数/points",
    "est_min_sessions": "等价同步/sessions",
    "push_velocity": "推分速度/velocity",
    "b50_mean_ach": "B50均达成/mean%",
    "b50_min_ach": "B50地板达成/min%",
    "b50_std_ach": "B50达成标准差/std%",
    "b50_mean_ds": "B50均定数/mean ds",
    "b50_min_ra": "B50地板RA/min ra",
    "b50_sss_plus_n": "B50 SSS+数",
    "b50_fc_n": "B50 FC+数",
    "b50_ap_n": "B50 AP+数",
    "b50_remaster_n": "B50 紫谱数/Re:M",
    "b35_mean_ach": "B35均达成",
    "b15_mean_ach": "B15均达成",
    "b15_vs_b35_ach": "B15-B35达成差",
    "full_master_plus_share": "全量Master+占比",
    "full_mean_ds": "全量均定数",
    "full_sss_plus_share": "全量SSS+占比",
    "b50_ra_sum": "B50 RA和",
    "rating_vs_b50_gap": "Rating-B50缺口",
}

import sys

KAGGLE_DATASET = os.environ.get("KAGGLE_DATASET", "awmcteam/dx-2026-awmcbot")

CANDIDATES = [
    Path("/kaggle/input"),
    Path("/content/drive/MyDrive/maimaidx_dataset"),
    Path("/content/maimaidx_dataset"),
    Path("/content/hf_upload"),
    Path("../data/hf_upload"),
    Path("data/hf_upload"),
    Path("./hf_upload"),
    Path("."),
]


def _root_from_players_jsonl(pth: Path) -> Path:
    return pth.parent.parent if pth.parent.name == "data" else pth.parent


def _search_players(base: Path, max_depth: int = 6):
    if not base.exists():
        return None
    for pth in (base / "data" / "players.jsonl", base / "players.jsonl"):
        if pth.exists():
            return _root_from_players_jsonl(pth)
    try:
        for pth in base.rglob("players.jsonl"):
            try:
                depth = len(pth.relative_to(base).parts)
            except ValueError:
                depth = 99
            if depth <= max_depth:
                return _root_from_players_jsonl(pth)
    except Exception:
        pass
    return None


def _try_kagglehub_download(slug: str):
    try:
        import kagglehub
    except ImportError:
        print("kagglehub missing; pip install ...")
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            import kagglehub
        except Exception as e:
            print("pip install kagglehub failed:", type(e).__name__, e)
            return None
    try:
        print("downloading dataset via kagglehub:", slug)
        path = Path(kagglehub.dataset_download(slug))
        print("kagglehub path:", path)
        hit = _search_players(path, max_depth=8)
        if hit:
            return hit
        for pth in path.rglob("players.jsonl"):
            return _root_from_players_jsonl(pth)
    except Exception as e:
        print("kagglehub download failed:", type(e).__name__, e)
        print("Colab: put kaggle.json in ~/.kaggle/ or set KAGGLE_USERNAME / KAGGLE_KEY")
        print("dataset:", "https://www.kaggle.com/datasets/" + slug)
    return None


def find_dataset_root() -> Path:
    for base in CANDIDATES:
        depth = 5 if str(base) == "/kaggle/input" else 3
        hit = _search_players(base, max_depth=depth)
        if hit:
            return hit

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        files = [f for f in kaggle_input.rglob("*") if f.is_file()]
        if files:
            print("Kaggle input files (sample):")
            for f in files[:40]:
                print(" ", f)
            hit = _search_players(kaggle_input, max_depth=8)
            if hit:
                return hit
        else:
            print("NOTE: /kaggle/input is empty. On Kaggle: Add Data -> awmcteam/dx-2026-awmcbot")

    hit = _try_kagglehub_download(KAGGLE_DATASET)
    if hit:
        return hit

    raise FileNotFoundError(
        "players.jsonl not found. "
        "Kaggle: Add Data awmcteam/dx-2026-awmcbot. "
        "Colab: configure Kaggle API then re-run this cell (auto kagglehub). "
        "Local: place hf_upload under data/hf_upload."
    )


ROOT = find_dataset_root()
DATA = ROOT / "data" if (ROOT / "data").exists() else ROOT
ASSETS = ROOT / "assets" if (ROOT / "assets").exists() else ROOT
if not (ASSETS / "arpi_peer_stats.json").exists():
    for alt in (
        ROOT / "arpi_peer_stats.json",
        DATA / "arpi_peer_stats.json",
        ROOT / "assets" / "arpi_peer_stats.json",
    ):
        if alt.exists():
            ASSETS = alt.parent
            break
print("ROOT =", ROOT)
print("DATA =", DATA)
print("ASSETS =", ASSETS)
print("MAX_STEP_DELTA =", MAX_STEP_DELTA, "(bulk if step exceeds)")
print("PLOT_FONT =", PLOT_FONT)
print("KAGGLE_DATASET =", KAGGLE_DATASET)


In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


meta_path = ROOT / "dataset_meta.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
print("dataset_meta:")
display(meta)

players_raw = read_jsonl(DATA / "players.jsonl")
trends = read_jsonl(DATA / "rating_trends.jsonl")
roast = read_jsonl(DATA / "roast_training_samples.jsonl")

peer_path = ASSETS / "arpi_peer_stats.json"
if not peer_path.exists():
    for alt in (ASSETS / "peer_stats.json", ROOT / "peer_stats.json"):
        if alt.exists():
            peer_path = alt
            break
peer = json.loads(peer_path.read_text(encoding="utf-8"))

print("players", len(players_raw), "trends", len(trends), "roast", len(roast))
print("peer buckets", len(peer.get("buckets") or {}))
players_raw.head(2)

## 1. 玩家汇总：增量同步 vs 累计上传

| 类型 | 含义 |
|------|------|
| `no_trend` | 趋势点 < 2 |
| `incremental` | 所有相邻步长 `|Δ| ≤ 337`（日常同步） |
| `bulk_upload` | 存在 `|Δ| > 337` 的步（累计游玩后统一上传） |
| `invalid` | rating 越界 / 无法解析（极少，才剔除趋势） |

In [ ]:
def trend_ratings(trend) -> list:
    out = []
    for p in trend or []:
        if not isinstance(p, dict):
            continue
        try:
            out.append(int(p.get("rating")))
        except (TypeError, ValueError):
            continue
    return out


def analyze_trend(trend) -> dict:
    ratings = trend_ratings(trend)
    steps = [ratings[i] - ratings[i - 1] for i in range(1, len(ratings))]
    if len(ratings) < 2:
        return {
            "trend_type": "no_trend",
            "trend_points": len(ratings),
            "rating_delta": None,
            "max_abs_step": None,
            "bulk_steps": 0,
            "incr_steps": 0,
            "bulk_delta_sum": 0,
            "incr_delta_sum": 0,
            "est_min_sessions": None,
            "invalid_reason": None,
        }

    invalid = None
    if any(r < RATING_MIN or r > RATING_MAX for r in ratings):
        invalid = f"rating_out_of_range[{min(ratings)},{max(ratings)}]"

    bulk_steps = [d for d in steps if abs(d) > MAX_STEP_DELTA]
    incr_steps = [d for d in steps if abs(d) <= MAX_STEP_DELTA]
    # 每一步至少覆盖 ceil(|d|/337) 次「满额同步」等价量
    est_sessions = int(sum(max(1, math.ceil(abs(d) / MAX_STEP_DELTA)) for d in steps))

    if invalid:
        ttype = "invalid"
    elif bulk_steps:
        ttype = "bulk_upload"
    else:
        ttype = "incremental"

    return {
        "trend_type": ttype,
        "trend_points": len(ratings),
        "rating_delta": ratings[-1] - ratings[0],
        "max_abs_step": max(abs(d) for d in steps),
        "bulk_steps": len(bulk_steps),
        "incr_steps": len(incr_steps),
        "bulk_delta_sum": int(sum(bulk_steps)),
        "incr_delta_sum": int(sum(incr_steps)),
        "est_min_sessions": est_sessions,
        "invalid_reason": invalid,
        "_steps": steps,
        "_ratings": ratings,
    }


def _as_float_list(items, key):
    out = []
    for x in items:
        if not isinstance(x, dict) or x.get(key) is None:
            continue
        try:
            out.append(float(x.get(key)))
        except (TypeError, ValueError):
            continue
    return out


def _fc_tier(fc) -> str:
    s = str(fc or "").lower().strip()
    if s in {"app", "ap+"}:
        return "app"
    if s in {"ap"}:
        return "ap"
    if s in {"fcp", "fc+"}:
        return "fcp"
    if s in {"fc"}:
        return "fc"
    return "none"


def flatten_player(row: dict) -> dict:
    latest = row.get("latest") or {}
    records = latest.get("records") or []
    b35 = latest.get("b35") or []
    b15 = latest.get("b15") or []
    b50 = b35 + b15
    ach = _as_float_list(b50, "achievements")
    ds_list = _as_float_list(b50, "ds")
    ra_list = _as_float_list(b50, "ra")
    b35_ach = _as_float_list(b35, "achievements")
    b15_ach = _as_float_list(b15, "achievements")

    rates = [norm_rate(x.get("rate")) for x in b50 if isinstance(x, dict)]
    fcs = [_fc_tier(x.get("fc")) for x in b50 if isinstance(x, dict)]
    li = []
    for x in b50:
        if not isinstance(x, dict):
            continue
        try:
            li.append(int(x.get("level_index")))
        except (TypeError, ValueError):
            continue

    tr = analyze_trend(row.get("rating_trend"))
    steps = tr.pop("_steps", [])
    ratings = tr.pop("_ratings", [])

    # 仅 invalid 才丢掉趋势字段；bulk 完整保留
    keep_trend = tr["trend_type"] != "invalid"
    est = tr.get("est_min_sessions")
    delta = tr.get("rating_delta")
    velocity = None
    if keep_trend and est and est > 0 and delta is not None:
        velocity = float(delta) / float(est)

    # 全量成绩轻量统计（有 records 时）
    full_master_plus = None
    full_mean_ds = None
    full_sss_plus = None
    if records:
        f_li, f_ds, f_rate = [], [], []
        for x in records:
            if not isinstance(x, dict):
                continue
            try:
                f_li.append(int(x.get("level_index")))
            except (TypeError, ValueError):
                pass
            try:
                if x.get("ds") is not None:
                    f_ds.append(float(x.get("ds")))
            except (TypeError, ValueError):
                pass
            f_rate.append(norm_rate(x.get("rate")))
        nrec = len(records)
        full_master_plus = (sum(1 for v in f_li if v >= 3) / nrec) if nrec else None
        full_mean_ds = float(np.mean(f_ds)) if f_ds else None
        full_sss_plus = (sum(1 for r in f_rate if r == "SSS+") / nrec) if nrec else None

    return {
        "player_id": row.get("player_id"),
        "rating": latest.get("rating"),
        "rating_bucket": latest.get("rating_bucket"),
        "arpi": latest.get("arpi"),
        "record_count": latest.get("record_count") or len(records),
        "b50_mode": latest.get("b50_mode"),
        "b35_n": len(b35),
        "b15_n": len(b15),
        "b50_ra_sum": latest.get("b50_ra_sum"),
        "rating_vs_b50_gap": latest.get("rating_vs_b50_gap"),
        "b50_mean_ach": float(np.mean(ach)) if ach else None,
        "b50_min_ach": float(np.min(ach)) if ach else None,
        "b50_std_ach": float(np.std(ach)) if len(ach) >= 2 else None,
        "b50_mean_ds": float(np.mean(ds_list)) if ds_list else None,
        "b50_min_ra": float(np.min(ra_list)) if ra_list else None,
        "b50_sss_plus_n": sum(1 for r in rates if r == "SSS+"),
        "b50_sss_n": sum(1 for r in rates if r in {"SSS", "SSS+"}),
        "b50_fc_n": sum(1 for f in fcs if f in {"fc", "fcp", "ap", "app"}),
        "b50_ap_n": sum(1 for f in fcs if f in {"ap", "app"}),
        "b50_remaster_n": sum(1 for v in li if v == 4),
        "b50_master_plus_n": sum(1 for v in li if v >= 3),
        "b35_mean_ach": float(np.mean(b35_ach)) if b35_ach else None,
        "b15_mean_ach": float(np.mean(b15_ach)) if b15_ach else None,
        "b15_vs_b35_ach": (
            float(np.mean(b15_ach) - np.mean(b35_ach))
            if b15_ach and b35_ach else None
        ),
        "full_master_plus_share": full_master_plus,
        "full_mean_ds": full_mean_ds,
        "full_sss_plus_share": full_sss_plus,
        "push_velocity": velocity,
        "date": latest.get("date"),
        **{k: (v if keep_trend or k in {"trend_type", "invalid_reason", "trend_points"} else None)
           for k, v in tr.items()},
        "_steps": steps if keep_trend else [],
        "_ratings": ratings if keep_trend else [],
    }


raw_records = players_raw.to_dict("records")
players = pd.DataFrame([flatten_player(r) for r in raw_records])
players = players.sort_values("rating", ascending=False).reset_index(drop=True)

print("趋势类型分布：")
display(players["trend_type"].value_counts(dropna=False))

invalid_n = int((players["trend_type"] == "invalid").sum())
bulk_n = int((players["trend_type"] == "bulk_upload").sum())
incr_n = int((players["trend_type"] == "incremental").sum())
print(
    f"incremental={incr_n}, bulk_upload={bulk_n} (保留), "
    f"invalid={invalid_n} (仅这些不参与趋势图)"
)
if invalid_n:
    display(players.loc[players["trend_type"] == "invalid", ["player_id", "rating", "invalid_reason"]].head(10))

players_trend = players[players["trend_type"].isin(["incremental", "bulk_upload"])].copy()
players_incr = players[players["trend_type"] == "incremental"].copy()
players_bulk = players[players["trend_type"] == "bulk_upload"].copy()
print(f"趋势可用样本: {len(players_trend)} / {len(players)}")

display(players.drop(columns=["_steps", "_ratings"], errors="ignore").describe(include="all").T)
players.drop(columns=["_steps", "_ratings"], errors="ignore").head(10)


# ===== 数据质量审计（防止评级/结构类低级错误）=====
def _audit_rates_from_raw(raw_records, max_players=300):
    from collections import Counter
    raw_c, norm_c = Counter(), Counter()
    n_b15_nonempty = 0
    n_players = 0
    modes = Counter()
    for row in raw_records[:max_players]:
        latest = row.get("latest") or {}
        b50 = (latest.get("b35") or []) + (latest.get("b15") or [])
        if latest.get("b15"):
            n_b15_nonempty += 1
        modes[str(latest.get("b50_mode"))] += 1
        for x in b50:
            if not isinstance(x, dict):
                continue
            raw_c[str(x.get("rate") or "")] += 1
            norm_c[norm_rate(x.get("rate"))] += 1
        n_players += 1
    return raw_c, norm_c, n_b15_nonempty, n_players, modes


raw_c, norm_c, n_b15_nonempty, n_aud, modes = _audit_rates_from_raw(raw_records)
print("=== DATA AUDIT ===")
print(f"players={len(players)}, audit_sample={n_aud}")
print("raw rate top:", raw_c.most_common(12))
print("norm_rate top:", norm_c.most_common(12))
sss_plus_players = int((players["b50_sss_plus_n"] > 0).sum()) if "b50_sss_plus_n" in players.columns else -1
print(
    f"players with b50_sss_plus_n>0: {sss_plus_players}/{len(players)} | "
    f"median SSS+ in B50: {players['b50_sss_plus_n'].median() if 'b50_sss_plus_n' in players.columns else 'NA'}"
)
print(f"players with nonempty b15 (sample): {n_b15_nonempty}/{n_aud}")
print("b50_mode counts (sample):", dict(modes))
if n_b15_nonempty == 0:
    print(
        "⚠ B15 全空：导出多半走了 top50_fallback（新老曲无法识别时 50 首全进 b35）。"
        "此时 b35_mean_ach ≈ b50_mean_ach，b15_* / 新曲压力 指标不可参考。"
    )
if sss_plus_players == 0:
    print("⚠ 全员 SSS+=0：rate 映射可能又坏了（应为 sssp→SSS+），请检查 norm_rate。")
else:
    print("✓ SSS+ 计数看起来正常（非全零）")


## 2. Rating / 同段桶 / ARPI / 成绩体量

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(players["rating"].dropna(), bins=30, color="#4C78A8")
axes[0, 0].set_title("Rating 分布 / Rating distribution")
axes[0, 0].set_xlabel("Rating / rating")

bucket_counts = (
    players["rating_bucket"].value_counts().rename_axis("bucket").reset_index(name="n")
)
bucket_counts["lo"] = bucket_counts["bucket"].str.split("-").str[0].astype(int)
bucket_counts = bucket_counts.sort_values("lo")
axes[0, 1].bar(bucket_counts["bucket"], bucket_counts["n"], color="#F58518")
axes[0, 1].set_title("各 Rating 桶人数 / Players per rating bucket")
axes[0, 1].tick_params(axis="x", rotation=75)

arpi = players["arpi"].dropna()
axes[1, 0].hist(arpi, bins=30, color="#54A24B")
axes[1, 0].axvline(0, color="black", lw=1)
axes[1, 0].set_title("玩家 ARPI 分布 / Player ARPI")
axes[1, 0].set_xlabel("ARPI / ARPI")

axes[1, 1].hist(players["record_count"].dropna(), bins=30, color="#B279A2")
axes[1, 1].set_title("全量成绩条数 / Full record_count")
axes[1, 1].set_xlabel("最新快照谱面数 / # charts in latest snapshot")

plt.tight_layout()
plt.show()

print("ARPI describe:")
display(arpi.describe())
print("B50 mean achievement describe:")
display(players["b50_mean_ach"].dropna().describe())


## 3. 推分趋势：保留 bulk，分层看

- **全量趋势样本**：含累计上传（大跳变）
- **incremental 子集**：日常同步，适合看「稳步推分」
- **bulk 子集**：看补存量级与等效 session 数

In [ ]:
print(f"有趋势样本: {len(players_trend)} / {len(players)}")
print("其中 incremental / bulk_upload:", incr_n, "/", bulk_n)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 类型占比
vc = players["trend_type"].value_counts()
axes[0, 0].bar(vc.index.astype(str), vc.values, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"][: len(vc)])
axes[0, 0].set_title("趋势类型计数 / trend_type counts")
axes[0, 0].tick_params(axis="x", rotation=20)

# 窗口 delta：全量（含 bulk）
d_all = players_trend["rating_delta"].dropna()
axes[0, 1].hist(d_all, bins=50, color="#E45756", alpha=0.9)
axes[0, 1].axvline(0, color="black", lw=1)
axes[0, 1].set_title("窗口 Rating 变化（含全部趋势） / window rating_delta (all trends)")
axes[0, 1].set_xlabel("末点 - 首点 / last - first")

# 散点：按类型着色
colors = {"incremental": "#72B7B2", "bulk_upload": "#E45756"}
for t, g in players_trend.groupby("trend_type"):
    axes[1, 0].scatter(
        g["rating"], g["rating_delta"],
        s=16, alpha=0.45, label=t, color=colors.get(t, "#999"),
    )
axes[1, 0].axhline(0, color="black", lw=1)
axes[1, 0].set_xlabel("Rating / rating")
axes[1, 0].set_ylabel("rating_delta / rating_delta")
axes[1, 0].set_title("ΔRating vs Rating（颜色=类型） / Delta vs rating (color=type)")
axes[1, 0].legend(fontsize=8)

# 等效最少 session
sess = players_trend["est_min_sessions"].dropna()
axes[1, 1].hist(sess, bins=40, color="#F58518")
axes[1, 1].set_title("估计最少同步等价次数 / est_min_sessions = sum ceil(|step|/337)")
axes[1, 1].set_xlabel("同步等价次数下界 / lower-bound sync equivalents")

plt.tight_layout()
plt.show()

print("window delta (all / incremental / bulk):")
display(pd.DataFrame({
    "all": d_all.describe(),
    "incremental": players_incr["rating_delta"].describe(),
    "bulk_upload": players_bulk["rating_delta"].describe(),
}))
print("est_min_sessions describe:")
display(sess.describe())


In [ ]:
# 相邻步长：0 太多会压扁细节 → 拆开看
all_steps = [d for steps in players_trend["_steps"] for d in steps]
step_s = pd.Series(all_steps, dtype=float)
incr_step_s = step_s[step_s.abs() <= MAX_STEP_DELTA]
bulk_step_s = step_s[step_s.abs() > MAX_STEP_DELTA]
incr_nz = incr_step_s[incr_step_s != 0]
incr_zero_n = int((incr_step_s == 0).sum())

print(
    f"incremental steps: n={len(incr_step_s)}, zero={incr_zero_n} "
    f"({incr_zero_n / max(len(incr_step_s),1):.1%}), nonzero={len(incr_nz)}"
)
print(f"bulk steps: n={len(bulk_step_s)}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) 全量 incremental：对数 y，才能同时看见 0 峰和非零尾巴
axes[0, 0].hist(incr_step_s, bins=67, color="#72B7B2", edgecolor="white", linewidth=0.3)
axes[0, 0].set_yscale("log")
axes[0, 0].axvline(0, color="black", lw=1)
axes[0, 0].axvline(MAX_STEP_DELTA, color="red", ls="--", lw=1)
axes[0, 0].axvline(-MAX_STEP_DELTA, color="red", ls="--", lw=1)
axes[0, 0].set_title(T("增量步长（对数纵轴）", f"incremental steps (log y) zero={incr_zero_n}"))
axes[0, 0].set_xlabel("单步 ΔRating / step dRating")

# 2) 非零细节（线性 y）
if len(incr_nz):
    axes[0, 1].hist(incr_nz, bins=67, color="#4C78A8", edgecolor="white", linewidth=0.3)
axes[0, 1].axvline(0, color="black", lw=1)
axes[0, 1].set_title("增量步长非零细节 / incremental steps nonzero only")
axes[0, 1].set_xlabel("单步 ΔRating（非零） / step dRating (!=0)")

# 3) bulk 长尾
if len(bulk_step_s):
    axes[1, 0].hist(bulk_step_s, bins=40, color="#E45756", edgecolor="white", linewidth=0.3)
    axes[1, 0].axvline(0, color="black", lw=1)
axes[1, 0].set_title(T("累计上传大步长", f"bulk steps (|d|>{MAX_STEP_DELTA})"))
axes[1, 0].set_xlabel("单步变化 / step delta")

# 4) bulk → 等效 session
if len(bulk_step_s):
    bulk_equiv = bulk_step_s.abs().map(lambda x: math.ceil(x / MAX_STEP_DELTA))
    axes[1, 1].hist(bulk_equiv, bins=min(30, max(5, bulk_equiv.nunique())), color="#F58518")
axes[1, 1].set_title("累计上传步 → 等价 session / bulk step → ceil(|d|/337)")
axes[1, 1].set_xlabel("每步等价 session 数 / equiv sessions per bulk step")

plt.tight_layout()
plt.show()

if len(incr_nz):
    print("nonzero incremental step describe:")
    display(incr_nz.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
    print("sign mix: + / - =", int((incr_nz > 0).sum()), "/", int((incr_nz < 0).sum()))

if len(bulk_step_s):
    print("largest bulk steps (cumulative uploads):")
    display(
        players_bulk.sort_values("max_abs_step", ascending=False)[
            ["player_id", "rating", "rating_delta", "max_abs_step", "bulk_steps", "est_min_sessions"]
        ].head(10)
    )

# 样本曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, subset, title in [
    (axes[0], players_incr, "incremental samples"),
    (axes[1], players_bulk, "bulk_upload samples"),
]:
    ids = subset.sort_values("trend_points", ascending=False).head(5)["player_id"].tolist()
    for pid in ids:
        row = players.loc[players["player_id"] == pid].iloc[0]
        ys = row["_ratings"]
        if len(ys) < 2:
            continue
        ax.plot(range(len(ys)), ys, marker="o", label=str(pid)[:8])
    ax.set_title(title)
    ax.set_xlabel("快照序号 / snapshot index")
    ax.set_ylabel("Rating / rating")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 4. 分段表现：同段桶 × 推分 / ARPI

按 rating 桶看窗口 Δ 与 ARPI；bulk 与 incremental 分开，避免累计上传拉高「日更」错觉。

In [ ]:
def bucket_lo(s):
    try:
        return int(str(s).split("-")[0])
    except Exception:
        return -1


g = players_trend.copy()
g["lo"] = g["rating_bucket"].map(bucket_lo)

by = (
    g.groupby(["rating_bucket", "trend_type"], dropna=False)
    .agg(
        n=("player_id", "count"),
        delta_mean=("rating_delta", "mean"),
        delta_median=("rating_delta", "median"),
        arpi_mean=("arpi", "mean"),
        sessions_mean=("est_min_sessions", "mean"),
        lo=("lo", "first"),
    )
    .reset_index()
)
# 统一桶顺序（按下界），后面所有图画线都按这个 x 位
bucket_order = (
    by[["rating_bucket", "lo"]].drop_duplicates().sort_values("lo")["rating_bucket"].tolist()
)
xpos = {b: i for i, b in enumerate(bucket_order)}
by["x"] = by["rating_bucket"].map(xpos)
by = by.sort_values(["trend_type", "x"])
display(by)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for t, color in [("incremental", "#72B7B2"), ("bulk_upload", "#E45756")]:
    sub = by[by["trend_type"] == t].sort_values("x")
    if sub.empty:
        continue
    # 用数值 x，避免 matplotlib 把字符串类别按「出现顺序」乱连线
    axes[0].plot(sub["x"], sub["delta_median"], marker="o", label=t, color=color, linewidth=1.5)
    axes[1].plot(sub["x"], sub["arpi_mean"], marker="o", label=t, color=color, linewidth=1.5)

for ax, title, ylabel in [
    (axes[0], "median window Δrating by bucket", "median Δrating"),
    (axes[1], "mean ARPI by bucket × trend_type", "mean ARPI"),
]:
    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(range(len(bucket_order)))
    # 桶太多时隔一个显示，避免挤成鬼画符
    step = 2 if len(bucket_order) > 18 else 1
    ax.set_xticks(range(0, len(bucket_order), step))
    ax.set_xticklabels([bucket_order[i] for i in range(0, len(bucket_order), step)], rotation=75)
    ax.set_xlim(-0.5, len(bucket_order) - 0.5)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()
plt.tight_layout()
plt.show()

# incremental 窗口 Δ 本身就很小；再给一张「非零 incremental」细节
incr_d = players_incr["rating_delta"].dropna()
incr_d_nz = incr_d[incr_d != 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(incr_d, bins=40, color="#72B7B2")
axes[0].set_yscale("log")
axes[0].axvline(0, color="black", lw=1)
axes[0].set_title("增量窗口 Δ（对数纵轴） / incremental window d (log y)")
if len(incr_d_nz):
    axes[1].hist(incr_d_nz, bins=40, color="#4C78A8")
axes[1].axvline(0, color="black", lw=1)
axes[1].set_title("增量窗口 Δ 非零细节 / incremental window d nonzero detail")
plt.tight_layout()
plt.show()

# ARPI vs Δ：incremental
if len(players_incr):
    fig, ax = plt.subplots(figsize=(7, 4))
    sc = ax.scatter(
        players_incr["arpi"],
        players_incr["rating_delta"],
        c=players_incr["rating"],
        cmap="viridis",
        s=18,
        alpha=0.55,
    )
    ax.axhline(0, color="black", lw=1)
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("ARPI / ARPI")
    ax.set_ylabel("窗口 rating_delta / window rating_delta")
    ax.set_title("仅增量：ARPI vs ΔRating / incremental only: ARPI vs dRating")
    plt.colorbar(sc, ax=ax, label="rating")
    plt.tight_layout()
    plt.show()


## 5. 同段 peer_stats（ARPI 桶）

In [ ]:
bucket_rows = []
for bkey, bval in (peer.get("buckets") or {}).items():
    dist = bval.get("arpi_distribution") or {}
    bucket_rows.append(
        {
            "bucket": bkey,
            "lo": int(str(bkey).split("-")[0]),
            "player_count": bval.get("player_count"),
            "chart_n": len(bval.get("charts") or {}),
            "arpi_mean": dist.get("mean"),
            "arpi_median": dist.get("median"),
            "arpi_p25": dist.get("p25"),
            "arpi_p75": dist.get("p75"),
            "arpi_count": dist.get("count"),
        }
    )

peer_df = pd.DataFrame(bucket_rows).sort_values("lo").reset_index(drop=True)
display(peer_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(peer_df["bucket"], peer_df["player_count"], color="#4C78A8")
axes[0].set_title("同段桶人数 / peer_stats player_count")
axes[0].tick_params(axis="x", rotation=75)

axes[1].plot(peer_df["bucket"], peer_df["arpi_median"], marker="o", label="median")
axes[1].fill_between(
    range(len(peer_df)),
    peer_df["arpi_p25"].astype(float),
    peer_df["arpi_p75"].astype(float),
    alpha=0.2,
    label="p25-p75",
)
axes[1].set_xticks(range(len(peer_df)))
axes[1].set_xticklabels(peer_df["bucket"], rotation=75)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("各桶 ARPI 分布 / ARPI distribution by bucket")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. 单桶热门上分谱（B50 出现率）

In [ ]:
def top_charts_in_bucket(bucket_key: str, top_n: int = 15) -> pd.DataFrame:
    charts = ((peer.get("buckets") or {}).get(bucket_key) or {}).get("charts") or {}
    rows = []
    for key, stat in charts.items():
        song_id, level_index = key.split(":", 1)
        rows.append(
            {
                "chart_key": key,
                "song_id": int(song_id),
                "level_index": int(level_index),
                "avg_achievement": stat.get("avg_achievement"),
                "sample_count": stat.get("sample_count"),
                "b50_appear_rate": stat.get("b50_appear_rate"),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["b50_appear_rate", "sample_count"], ascending=False).head(top_n)


focus_bucket = peer_df.sort_values("player_count", ascending=False).iloc[0]["bucket"]
print("focus bucket =", focus_bucket)
top_charts = top_charts_in_bucket(focus_bucket)
display(top_charts)

if not top_charts.empty:
    plt.figure(figsize=(10, 5))
    plt.barh(top_charts["chart_key"][::-1], top_charts["b50_appear_rate"][::-1], color="#B279A2")
    plt.xlabel("b50_appear_rate")
    plt.title(T(f"桶内 B50 热门谱 ({focus_bucket})", f"Top B50 charts in {focus_bucket}"))
    plt.tight_layout()
    plt.show()


## 7. B50 结构可视化（谱面级展开）

从每位玩家的 B35/B15 展开成长表，看难度色、定数、达成率档、FC/FS 等。


In [ ]:
# 展开 B50 为长表（可能几十万行，抽样/聚合后再画）
DIFF = {0: "Basic", 1: "Advanced", 2: "Expert", 3: "Master", 4: "Re:Master"}

def iter_b50_rows(raw_records, max_players=None):
    n = 0
    for row in raw_records:
        latest = row.get("latest") or {}
        pid = row.get("player_id")
        rating = latest.get("rating")
        bucket = latest.get("rating_bucket")
        for part, arr in (("b35", latest.get("b35") or []), ("b15", latest.get("b15") or [])):
            for r in arr:
                if not isinstance(r, dict):
                    continue
                yield {
                    "player_id": pid,
                    "rating": rating,
                    "rating_bucket": bucket,
                    "part": part,
                    "song_id": r.get("song_id"),
                    "title": r.get("title") or "",
                    "level_index": r.get("level_index"),
                    "diff": DIFF.get(int(r.get("level_index") or -1), str(r.get("level_index"))),
                    "ds": r.get("ds"),
                    "achievements": r.get("achievements"),
                    "rate": norm_rate(r.get("rate")),
                    "ra": r.get("ra"),
                    "fc": r.get("fc") or "none",
                    "fs": r.get("fs") or "none",
                    "dxScore": r.get("dxScore"),
                }
        n += 1
        if max_players is not None and n >= max_players:
            break

b50_long = pd.DataFrame(list(iter_b50_rows(raw_records)))
print("b50_long rows:", len(b50_long), "players:", b50_long["player_id"].nunique())
display(b50_long.head(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# difficulty mix
diff_order = ["Basic", "Advanced", "Expert", "Master", "Re:Master"]
diff_vc = b50_long["diff"].value_counts().reindex(diff_order).fillna(0)
axes[0, 0].bar(diff_order, diff_vc.values, color=["#22c55e", "#f59e0b", "#ef4444", "#a855f7", "#e5e5e5"])
axes[0, 0].set_title("B50 难度构成 / B50 charts by difficulty")
axes[0, 0].tick_params(axis="x", rotation=20)

# B35 vs B15 数量（每人）
part_player = b50_long.groupby(["player_id", "part"]).size().unstack(fill_value=0)
axes[0, 1].hist(part_player.get("b35", pd.Series(dtype=float)), bins=20, alpha=0.7, label="b35", color="#4C78A8")
axes[0, 1].hist(part_player.get("b15", pd.Series(dtype=float)), bins=20, alpha=0.7, label="b15", color="#F58518")
axes[0, 1].set_title("每人 B35 / B15 谱数 / #charts per player in B35 / B15")
axes[0, 1].legend()

# ds distribution
ds = pd.to_numeric(b50_long["ds"], errors="coerce").dropna()
axes[1, 0].hist(ds, bins=40, color="#54A24B")
axes[1, 0].set_title("B50 定数分布 / B50 chart ds distribution")
axes[1, 0].set_xlabel("定数 ds / ds")

# achievement distribution
ach = pd.to_numeric(b50_long["achievements"], errors="coerce").dropna()
axes[1, 1].hist(ach, bins=40, color="#B279A2")
axes[1, 1].set_title("B50 达成率分布 / B50 achievements %")
axes[1, 1].set_xlabel("达成率 / achievement")
plt.tight_layout()
plt.show()

print("ds describe:"); display(ds.describe())
print("achievements describe:"); display(ach.describe())


In [ ]:
# rate / FC / FS
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rate_order = ["SSS+", "SSS", "SS+", "SS", "S+", "S", "AAA", "AA", "A", "B", "C", "D"]
rate_vc = b50_long["rate"].value_counts()
rate_vc = rate_vc.reindex([r for r in rate_order if r in rate_vc.index]).fillna(0)
# 其余少见档位并到 OTHER
other = int(b50_long["rate"].value_counts().sum() - rate_vc.sum())
axes[0].bar(list(rate_vc.index) + (["OTHER"] if other else []), list(rate_vc.values) + ([other] if other else []), color="#4C78A8")
axes[0].set_title("B50 评级档位 / B50 rate tier")
axes[0].tick_params(axis="x", rotation=45)

fc_vc = b50_long["fc"].fillna("none").replace("", "none").value_counts()
axes[1].bar(fc_vc.index.astype(str), fc_vc.values, color="#F58518")
axes[1].set_title("B50 FC 状态 / FC status in B50")
axes[1].tick_params(axis="x", rotation=30)

fs_vc = b50_long["fs"].fillna("none").replace("", "none").value_counts()
axes[2].bar(fs_vc.index.astype(str), fs_vc.values, color="#54A24B")
axes[2].set_title("B50 FS 状态 / FS status in B50")
axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# 分 part 的 rate 堆叠（比例）
ct = pd.crosstab(b50_long["part"], b50_long["rate"], normalize="index")
# 只保留常见档
keep = [r for r in rate_order if r in ct.columns]
ct = ct[keep] if keep else ct
ct.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="viridis")
plt.title("评级构成：B35 vs B15（行%） / rate mix: B35 vs B15 (row %)")
plt.ylabel("share")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 热门曲目（按 B50 出现次数）——曲名用 CJK 字体；失败则显示 #song_id
title_vc = (
    b50_long.groupby(["song_id", "title", "diff"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
print("Top titles in B50:")
display(title_vc.head(20))

top20 = title_vc.head(20).iloc[::-1].copy()
top20["label"] = [
    chart_axis_label(t, d, sid, PLOT_FONT, max_len=34)
    for t, d, sid in zip(top20["title"], top20["diff"], top20["song_id"])
]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top20["label"], top20["size"], color="#B279A2")
ax.set_xlabel("在所有 B50 中出现次数 / appearances in all B50s")
ax.set_title("出现次数 Top 20 谱面 / Top 20 chart appearances")
# 数值标注，细节更清晰
for bar, val in zip(bars, top20["size"]):
    ax.text(val + max(top20["size"]) * 0.01, bar.get_y() + bar.get_height() / 2,
            str(int(val)), va="center", fontsize=8)
ax.set_xlim(0, max(top20["size"]) * 1.12)
plt.tight_layout()
plt.show()

# 单谱平均达成率：B50 本身都是高分，0-100 轴几乎看不出差别。
# 这里只看 Master+，并把 x 轴缩到数据范围；「低」侧才是相对难啃的上分谱。
chart_stats = (
    b50_long.groupby(["song_id", "title", "diff"], as_index=False)
    .agg(
        n=("achievements", "count"),
        avg_ach=("achievements", "mean"),
        avg_ra=("ra", "mean"),
        avg_ds=("ds", "mean"),
    )
)
chart_stats = chart_stats[chart_stats["n"] >= 30].copy()
masterish = chart_stats[chart_stats["diff"].isin(["Master", "Re:Master"])].copy()
masterish["label"] = [
    chart_axis_label(t, d, sid, PLOT_FONT, max_len=34)
    for t, d, sid in zip(masterish["title"], masterish["diff"], masterish["song_id"])
]

print("Master+ charts avg achievement range:",
      float(masterish["avg_ach"].min()), "~", float(masterish["avg_ach"].max()),
      f"(n_charts={len(masterish)})")
print("Highest avg achievement Master+ (n>=30):")
display(masterish.sort_values("avg_ach", ascending=False).head(15))
print("Lowest avg achievement Master+ (n>=30):  # 相对难 / 常被低达成塞进 B50")
display(masterish.sort_values("avg_ach").head(15))

hi = masterish.sort_values("avg_ach", ascending=False).head(12).iloc[::-1]
lo = masterish.sort_values("avg_ach").head(12).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title, color in [
    (axes[0], hi, T("Master+ 平均达成率最高（n≥30）", "Highest avg ach Master+ (n>=30)"), "#54A24B"),
    (axes[1], lo, T("Master+ 平均达成率最低（n≥30）", "Lowest avg ach Master+ (n>=30)"), "#E45756"),
]:
    bars = ax.barh(df["label"], df["avg_ach"], color=color)
    # 缩放到数据附近，否则全是贴满 100 的长条
    lo_x = float(df["avg_ach"].min()) - 0.15
    hi_x = min(101.0, float(df["avg_ach"].max()) + 0.05)
    ax.set_xlim(lo_x, hi_x)
    for bar, val, n in zip(bars, df["avg_ach"], df["n"]):
        ax.text(val, bar.get_y() + bar.get_height() / 2,
                f" {val:.3f}% (n={int(n)})", va="center", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("平均达成率 %（放大坐标） / avg achievement % (zoomed)")
plt.tight_layout()
plt.show()


## 8. 玩家画像：相关、箱线、CDF、缺口


In [ ]:
# 数值特征相关矩阵（可读版）
# - 去掉常数列 / 全 NaN 列（否则整行 nan，还特别挤）
# - 若 B15 全空，去掉与 B50 重复的 b35/b15 指标
# - 只保留「有区分度」的核心列，放大画布、稀疏标注

candidate_cols = [
    "rating", "arpi", "record_count", "rating_delta", "trend_points",
    "est_min_sessions", "push_velocity",
    "b50_mean_ach", "b50_min_ach", "b50_std_ach", "b50_mean_ds", "b50_min_ra",
    "b50_sss_plus_n", "b50_fc_n", "b50_ap_n", "b50_remaster_n",
    "b35_mean_ach", "b15_mean_ach", "b15_vs_b35_ach",
    "full_master_plus_share", "full_mean_ds", "full_sss_plus_share",
    "b50_ra_sum", "rating_vs_b50_gap",
]
candidate_cols = [c for c in candidate_cols if c in players.columns]

# B15 基本不可用时，相关矩阵别塞重复/空列
b15_usable = (
    "b15_mean_ach" in players.columns
    and players["b15_mean_ach"].notna().mean() > 0.2
    and players.get("b15_n", pd.Series(dtype=float)).fillna(0).gt(0).mean() > 0.2
) if "b15_n" in players.columns else (
    "b15_mean_ach" in players.columns and players["b15_mean_ach"].notna().mean() > 0.2
)
if not b15_usable:
    drop_b15 = {"b15_mean_ach", "b15_vs_b35_ach", "b35_mean_ach"}
    candidate_cols = [c for c in candidate_cols if c not in drop_b15]
    print("相关矩阵：已去掉 b15/b35 达成率列（B15 样本不足，避免假相关=1.00）")

feat = players[candidate_cols].apply(pd.to_numeric, errors="coerce")
keep = []
dropped = []
for c in feat.columns:
    s = feat[c]
    if s.notna().sum() < max(30, int(0.1 * len(s))):
        dropped.append((c, "too_few_non_null"))
        continue
    if s.dropna().nunique() <= 1:
        dropped.append((c, "constant_or_single_value"))
        continue
    # 标准差过小也没参考价值
    if float(s.std(skipna=True) or 0) < 1e-12:
        dropped.append((c, "near_zero_variance"))
        continue
    keep.append(c)

print("correlation dropped:", dropped)
print("correlation kept:", keep)

corr = feat[keep].corr(numeric_only=True)
# 短标签
labels = [FEATURE_LABELS.get(c, c) for c in corr.columns]

n = len(corr)
fig_w = max(10, 0.7 * n + 3)
fig_h = max(8, 0.65 * n + 2)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

# 只标注 |r|>=0.25，对角线不标，减轻拥挤
for i in range(n):
    for j in range(n):
        if i == j:
            continue
        v = corr.values[i, j]
        if np.isnan(v) or abs(v) < 0.25:
            continue
        ax.text(
            j, i, f"{v:.2f}",
            ha="center", va="center",
            fontsize=8,
            color=("white" if abs(v) >= 0.75 else "black"),
        )

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label("Pearson r")
ax.set_title(T("玩家特征相关矩阵（|r|≥0.25 才标数字）", "Player feature correlation"))
plt.tight_layout()
plt.show()

# 额外给出「与 rating 相关」排序表，比热力图更好读
if "rating" in corr.columns:
    rel = corr["rating"].drop(labels=["rating"]).sort_values(key=lambda s: s.abs(), ascending=False)
    rel_df = pd.DataFrame({
        "feature": rel.index,
        "label": [FEATURE_LABELS.get(i, i) for i in rel.index],
        "corr_with_rating": rel.values,
    })
    print("与 Rating 相关性（按绝对值）:")
    display(rel_df)

# Rating CDF
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
r = players["rating"].dropna().sort_values()
axes[0].plot(r.values, np.linspace(0, 1, len(r)), color="#4C78A8")
axes[0].set_title(T("Rating 累积分布", "Rating CDF"))
axes[0].set_xlabel(T("Rating", "rating"))
axes[0].set_ylabel(T("分位", "quantile"))

if "b50_ra_sum" in players.columns:
    gap = players["rating"] - players["b50_ra_sum"]
    axes[1].hist(gap.dropna(), bins=40, color="#E45756")
    axes[1].axvline(0, color="black", lw=1)
    axes[1].set_title(T("Rating − B50 RA 和（缺口）", "rating - b50_ra_sum (gap)"))
    axes[1].set_xlabel(T("缺口", "gap"))
    print("gap describe:"); display(gap.dropna().describe())
else:
    axes[1].text(0.5, 0.5, "no b50_ra_sum in dataset", ha="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
import warnings
# ARPI / delta 箱线图 by bucket；record_count vs rating
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tmp = players.dropna(subset=["arpi", "rating_bucket"]).copy()
tmp["lo"] = tmp["rating_bucket"].map(bucket_lo)
tmp = tmp.sort_values("lo")
buckets = tmp["rating_bucket"].unique().tolist()
pairs = [(b, tmp.loc[tmp["rating_bucket"] == b, "arpi"].dropna().values) for b in buckets]
pairs = [(b, v) for b, v in pairs if len(v)]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Mean of empty slice")
    axes[0].boxplot(
        [v for _, v in pairs],
        tick_labels=[b for b, _ in pairs],
        showfliers=False,
    )
axes[0].axhline(0, color="black", lw=1)
axes[0].tick_params(axis="x", rotation=75)
axes[0].set_title("各桶 ARPI 箱线（隐藏离群） / ARPI boxplot by bucket (no outliers)")

# delta box by trend_type
td = players_trend.dropna(subset=["rating_delta"])
types = ["incremental", "bulk_upload"]
delta_pairs = [
    (t, td.loc[td["trend_type"] == t, "rating_delta"].dropna().values)
    for t in types
]
delta_pairs = [(t, v) for t, v in delta_pairs if len(v)]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Mean of empty slice")
    axes[1].boxplot(
        [v for _, v in delta_pairs],
        tick_labels=[t for t, _ in delta_pairs],
        showfliers=True,
    )
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("窗口 ΔRating（按趋势类型） / window dRating by trend_type")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(players["rating"], players["record_count"], s=12, alpha=0.35, color="#4C78A8")
axes[0].set_xlabel("Rating / rating"); axes[0].set_ylabel("成绩条数 / record_count")
axes[0].set_title("Rating vs 全量谱数 / rating vs full record_count")

axes[1].scatter(players["rating"], players["b50_mean_ach"], s=12, alpha=0.35, c=players["arpi"], cmap="coolwarm")
axes[1].set_xlabel("Rating / rating"); axes[1].set_ylabel("B50 平均达成率 / b50_mean_ach")
axes[1].set_title("Rating vs B50 均达成（色=ARPI） / rating vs B50 mean ach (color=ARPI)")
plt.tight_layout()
plt.show()


In [ ]:
# bucket × rate heatmap (B50 charts)
tmp = b50_long.copy()
tmp["lo"] = tmp["rating_bucket"].map(bucket_lo)
tmp = tmp.dropna(subset=["rating_bucket"])
rate_keep = [r for r in ["SSS+", "SSS", "SS+", "SS", "S+", "S"] if r in set(tmp["rate"])]
# 按桶下界排序的 bucket 列表
b_order = (
    tmp[["rating_bucket", "lo"]].drop_duplicates().sort_values("lo")["rating_bucket"].tolist()
)
heat = pd.crosstab(tmp["rating_bucket"], tmp["rate"])
heat = heat.reindex(index=b_order)
heat = heat[[c for c in rate_keep if c in heat.columns]]
# 行归一化
heat_pct = heat.div(heat.sum(axis=1).replace(0, np.nan), axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heat_pct.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat_pct.columns)))
ax.set_xticklabels(heat_pct.columns)
ax.set_yticks(range(len(heat_pct.index)))
ax.set_yticklabels(heat_pct.index)
ax.set_title("各桶 B50 评级构成（行%） / B50 rate mix by rating bucket (row %)")
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.show()

# difficulty × bucket (Master / Re:Master share)
diff_heat = pd.crosstab(tmp["rating_bucket"], tmp["diff"], normalize="index")
diff_heat = diff_heat.reindex(index=b_order)
cols = [c for c in ["Expert", "Master", "Re:Master"] if c in diff_heat.columns]
if cols:
    diff_heat[cols].plot(kind="area", stacked=True, figsize=(10, 4), alpha=0.8)
    plt.title("各桶 B50 难度构成（占比） / B50 difficulty mix by bucket (share)")
    plt.ylabel("share")
    plt.xticks(rotation=75)
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


## 9. 锐评样本分布


In [ ]:
# roast feasibility / 结构
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "push_feasibility_hint" in roast.columns:
    vc = roast["push_feasibility_hint"].value_counts()
    axes[0].barh(vc.index.astype(str)[::-1], vc.values[::-1], color="#4C78A8")
    axes[0].set_title("推分可行性提示 / push_feasibility_hint")
if "rating_delta" in roast.columns:
    axes[1].hist(pd.to_numeric(roast["rating_delta"], errors="coerce").dropna(), bins=40, color="#E45756")
    axes[1].axvline(0, color="black", lw=1)
    axes[1].set_title("锐评样本 rating_delta / roast.rating_delta")
plt.tight_layout()
plt.show()

if "player_id" in roast.columns:
    roast2 = roast.copy()
    roast2["trend_type"] = roast2["player_id"].map(players.set_index("player_id")["trend_type"].to_dict())
    if "push_feasibility_hint" in roast2.columns:
        ct = pd.crosstab(roast2["trend_type"], roast2["push_feasibility_hint"])
        display(ct)
        ct.plot(kind="bar", stacked=True, figsize=(9, 4))
        plt.title("可行性提示 × 趋势类型 / feasibility hint x trend_type")
        plt.xticks(rotation=20)
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        plt.tight_layout()
        plt.show()


## 10. 更多可统计指标

在已有 Rating / ARPI / B50 热门外，补充一批**玩家层 + 谱面层**统计，便于同段对比与锐评素材：

| 指标 | 含义 |
|------|------|
| `b50_fc_n` / `b50_ap_n` | B50 中 FC 及以上 / AP 及以上张数 |
| `b50_sss_plus_n` | B50 中 SSS+ 张数 |
| `b50_mean_ds` / `b50_std_ach` | B50 平均定数、达成率离散（是否「高低不一」） |
| `b50_min_ra` / `b50_min_ach` | B50 地板（拖后腿谱） |
| `b35_mean_ach` vs `b15_mean_ach` | 旧曲榜 vs 新曲榜压力差 |
| `push_velocity` | `rating_delta / est_min_sessions`（增量同步推分效率） |
| `full_*` | 全量成绩：Master+ 占比、平均定数、SSS+ 占比 |


In [ ]:
# ---- 玩家层衍生指标汇总（flatten 已算好）----
extra_cols = [
    "b50_sss_plus_n", "b50_fc_n", "b50_ap_n", "b50_remaster_n",
    "b50_mean_ds", "b50_std_ach", "b50_min_ra",
    "b35_mean_ach", "b15_mean_ach", "b15_vs_b35_ach", "push_velocity",
    "full_master_plus_share", "full_mean_ds", "full_sss_plus_share",
]
print("extra feature describe:")
display(players[extra_cols].describe().T)

# 新曲压力：b15 平均达成率相对 b35（flatten 已算 b15_vs_b35_ach）
if "b15_vs_b35_ach" in players.columns and players["b15_vs_b35_ach"].notna().any():
    print("b15 - b35 mean achievement (neg => newer charts harder / lower):")
    display(players["b15_vs_b35_ach"].describe())
else:
    print("⚠ 跳过新曲压力：b15_vs_b35_ach 全空（数据集 B15 未拆分）")

# 按桶聚合中位数
tmp = players.dropna(subset=["rating_bucket"]).copy()
tmp["lo"] = tmp["rating_bucket"].map(bucket_lo)
tmp = tmp.sort_values("lo")
bucket_order = list(dict.fromkeys(tmp.sort_values("lo")["rating_bucket"]))

def bucket_median(col):
    # 全 NaN 桶不要走 nanmedian，避免 Mean of empty slice 警告
    s = tmp.groupby("rating_bucket")[col].apply(
        lambda x: float(x.dropna().median()) if x.dropna().size else np.nan
    )
    return s.reindex(bucket_order)

agg = pd.DataFrame({
    "n": tmp.groupby("rating_bucket").size().reindex(bucket_order),
    "sss+_med": bucket_median("b50_sss_plus_n"),
    "fc_med": bucket_median("b50_fc_n"),
    "ap_med": bucket_median("b50_ap_n"),
    "mean_ds_med": bucket_median("b50_mean_ds"),
    "std_ach_med": bucket_median("b50_std_ach"),
    "min_ra_med": bucket_median("b50_min_ra"),
    "b15-b35_med": bucket_median("b15_vs_b35_ach"),
    "velocity_med": bucket_median("push_velocity"),
    "full_master+_med": bucket_median("full_master_plus_share"),
})
print("bucket medians:")
display(agg)

x = np.arange(len(bucket_order))
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(x, agg["sss+_med"], "o-", label="SSS+", color="#4C78A8")
axes[0, 0].plot(x, agg["fc_med"], "s-", label="FC+", color="#54A24B")
axes[0, 0].plot(x, agg["ap_med"], "^-", label="AP+", color="#E45756")
axes[0, 0].set_xticks(x[:: max(1, len(x)//8)])
axes[0, 0].set_xticklabels([bucket_order[i] for i in range(0, len(x), max(1, len(x)//8))], rotation=75)
axes[0, 0].set_title("各桶 B50 奖牌数（中位数） / B50 medal counts by bucket (median)")
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(x, agg["mean_ds_med"], "o-", color="#F58518")
axes[0, 1].set_xticks(x[:: max(1, len(x)//8)])
axes[0, 1].set_xticklabels([bucket_order[i] for i in range(0, len(x), max(1, len(x)//8))], rotation=75)
axes[0, 1].set_title("各桶 B50 平均定数（中位数） / B50 mean ds by bucket (median)")
axes[0, 1].set_ylabel("定数 ds / ds")

axes[1, 0].plot(x, agg["min_ra_med"], "o-", color="#B279A2", label="min ra")
axes[1, 0].plot(x, agg["std_ach_med"], "s-", color="#72B7B2", label="ach std")
axes[1, 0].set_xticks(x[:: max(1, len(x)//8)])
axes[1, 0].set_xticklabels([bucket_order[i] for i in range(0, len(x), max(1, len(x)//8))], rotation=75)
axes[1, 0].set_title("B50 地板(min ra)与不均衡(ach std) / B50 floor & unevenness")
axes[1, 0].legend(fontsize=8)

if agg["b15-b35_med"].notna().any():
    axes[1, 1].plot(x, agg["b15-b35_med"], "o-", color="#E45756", label="b15-b35 ach")
    axes[1, 1].axhline(0, color="black", lw=1)
    axes[1, 1].legend(fontsize=8)
else:
    axes[1, 1].text(0.5, 0.5, "B15 unavailable / B15 不可用", transform=axes[1, 1].transAxes, ha="center")
axes[1, 1].set_xticks(x[:: max(1, len(x)//8)])
axes[1, 1].set_xticklabels([bucket_order[i] for i in range(0, len(x), max(1, len(x)//8))], rotation=75)
axes[1, 1].set_title("新曲压力：b15−b35 均达成 / New-chart pressure: b15 - b35 mean ach")

plt.tight_layout()
plt.show()


In [ ]:
# ---- 推分效率 / 全量谱面 / 拖后腿谱 ----
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1) incremental 推分速度
incr_v = players_incr["push_velocity"].dropna()
axes[0, 0].hist(incr_v, bins=40, color="#4C78A8")
axes[0, 0].axvline(incr_v.median(), color="red", lw=1, label=f"median={incr_v.median():.2f}")
axes[0, 0].set_title("推分速度（仅增量同步） / push_velocity (incremental only)")
axes[0, 0].set_xlabel("rating_delta / 估计同步次数 / rating_delta / est_min_sessions")
axes[0, 0].legend(fontsize=8)

# 2) rating vs velocity
sc = players_incr.dropna(subset=["push_velocity", "rating"])
axes[0, 1].scatter(sc["rating"], sc["push_velocity"], s=12, alpha=0.35, c=sc.get("arpi", 0), cmap="coolwarm")
axes[0, 1].axhline(0, color="black", lw=1)
axes[0, 1].set_xlabel("Rating / rating")
axes[0, 1].set_ylabel("推分速度 / push_velocity")
axes[0, 1].set_title("Rating vs 推分速度（色≈ARPI） / rating vs push_velocity (color~ARPI)")

# 3) full catalog Master+ share vs rating
full = players.dropna(subset=["full_master_plus_share", "rating"])
if len(full):
    axes[1, 0].scatter(full["rating"], full["full_master_plus_share"], s=12, alpha=0.35, color="#54A24B")
    axes[1, 0].set_xlabel("Rating / rating")
    axes[1, 0].set_ylabel("全量中 Master+ 占比 / Master+ share in full records")
    axes[1, 0].set_title("全量成绩：Master+ 渗透率 / full records: Master+ penetration")
else:
    axes[1, 0].text(0.5, 0.5, "no full records in dataset", ha="center")
    axes[1, 0].set_axis_off()

# 4) SSS+ count vs rating
axes[1, 1].scatter(
    players["rating"], players["b50_sss_plus_n"],
    s=12, alpha=0.35, c=players.get("b50_ap_n", 0), cmap="viridis"
)
axes[1, 1].set_xlabel("Rating / rating")
axes[1, 1].set_ylabel("B50 SSS+ 张数 / B50 SSS+ count")
axes[1, 1].set_title("B50 SSS+ 张数 vs Rating（色=AP数） / B50 SSS+ vs rating (color=AP count)")
plt.tight_layout()
plt.show()

# 每人 B50 中 ra 最低的谱 = 拖后腿；聚合看「常见地板谱」
_ra = pd.to_numeric(b50_long["ra"], errors="coerce")
b50_ra = b50_long.assign(_ra=_ra).dropna(subset=["_ra"])
idx = b50_ra.groupby("player_id")["_ra"].idxmin()
floor_charts = b50_ra.loc[idx].copy()
floor_vc = (
    floor_charts.groupby(["song_id", "title", "diff"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
print("Most common B50 floor charts (lowest ra per player):")
display(floor_vc.head(20))

top_floor = floor_vc.head(15).iloc[::-1].copy()
top_floor["label"] = [
    chart_axis_label(t, d, sid, PLOT_FONT, max_len=34)
    for t, d, sid in zip(top_floor["title"], top_floor["diff"], top_floor["song_id"])
]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top_floor["label"], top_floor["size"], color="#E45756")
for bar, val in zip(bars, top_floor["size"]):
    ax.text(val + max(top_floor["size"]) * 0.01, bar.get_y() + bar.get_height() / 2,
            str(int(val)), va="center", fontsize=8)
ax.set_xlabel("以该谱为 B50 RA 地板的玩家数 / #players with this B50 ra floor")
ax.set_title("B50 拖后腿谱 Top 15 / Top 15 B50 floor charts")
ax.set_xlim(0, max(top_floor["size"]) * 1.12)
plt.tight_layout()
plt.show()

# FC/AP 等级结构（B50 全体）
fc_order = ["none", "fc", "fcp", "ap", "app"]
fc_vc = b50_long["fc"].map(lambda x: _fc_tier(x)).value_counts().reindex(fc_order).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(fc_order, fc_vc.values, color="#4C78A8")
axes[0].set_title("B50 FC 档位构成 / B50 FC tier mix")
axes[0].set_ylabel("谱面数 / charts")

# 定数带 × 平均达成率
b50_long["_ds"] = pd.to_numeric(b50_long["ds"], errors="coerce")
b50_long["_ach"] = pd.to_numeric(b50_long["achievements"], errors="coerce")
ds_ok = b50_long.dropna(subset=["_ds", "_ach"])
ds_bin = pd.cut(ds_ok["_ds"], bins=np.arange(10, 15.5, 0.5), right=False)
ds_ach = ds_ok.groupby(ds_bin, observed=True)["_ach"].agg(["mean", "count"])
ds_ach = ds_ach[ds_ach["count"] >= 30]
if len(ds_ach):
    axes[1].plot(range(len(ds_ach)), ds_ach["mean"].values, "o-", color="#F58518")
    axes[1].set_xticks(range(len(ds_ach)))
    axes[1].set_xticklabels([str(i) for i in ds_ach.index], rotation=60)
    axes[1].set_title("各定数带平均达成率（n≥30） / avg achievement by ds band (n>=30)")
    axes[1].set_ylabel("平均达成率 % / avg achievement %")
else:
    axes[1].text(0.5, 0.5, "not enough ds-band samples", ha="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

# Remaster 渗透：各桶 B50 中 Re:Master 平均张数
rem = tmp.groupby("rating_bucket")["b50_remaster_n"].mean().reindex(bucket_order)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(len(bucket_order)), rem.values, color="#a855f7")
ax.set_xticks(range(0, len(bucket_order), max(1, len(bucket_order)//10)))
ax.set_xticklabels([bucket_order[i] for i in range(0, len(bucket_order), max(1, len(bucket_order)//10))], rotation=75)
ax.set_ylabel("B50 中 Re:Master 平均张数 / avg Re:Master charts in B50")
ax.set_title("各桶 Re:Master 渗透 / Re:Master penetration by rating bucket")
plt.tight_layout()
plt.show()


## 11. 导出工作区结果

导出全量汇总 + incremental / bulk 子集，便于在 Kaggle 外继续建模。


In [ ]:
# 导出前按最新 players 刷新子集，避免中途新增列后对不齐
players_trend = players[players["trend_type"].isin(["incremental", "bulk_upload"])].copy()
players_incr = players[players["trend_type"] == "incremental"].copy()
players_bulk = players[players["trend_type"] == "bulk_upload"].copy()

export_cols = [c for c in players.columns if not c.startswith("_")]
players_export = players[export_cols].copy()

def _cols_in(df, cols):
    return [c for c in cols if c in df.columns]

# 给 roast 挂上 trend_type，便于分层提示词
type_map = players.set_index("player_id")["trend_type"].to_dict()
roast_x = roast.copy()
if "player_id" in roast_x.columns:
    roast_x["trend_type"] = roast_x["player_id"].map(type_map)

cols = [c for c in ["player_id", "rating", "rating_bucket", "arpi", "rating_delta", "trend_type", "push_feasibility_hint"] if c in roast_x.columns]
display(roast_x[cols].head(10))
if "push_feasibility_hint" in roast_x.columns:
    print("feasibility x trend_type:")
    display(pd.crosstab(roast_x.get("trend_type"), roast_x["push_feasibility_hint"]))

out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
players_export.to_csv(out_dir / "players_summary.csv", index=False)
players_trend[_cols_in(players_trend, export_cols)].to_csv(out_dir / "players_trend_all.csv", index=False)
players_incr[_cols_in(players_incr, export_cols)].to_csv(out_dir / "players_trend_incremental.csv", index=False)
players_bulk[_cols_in(players_bulk, export_cols)].to_csv(out_dir / "players_trend_bulk_upload.csv", index=False)
peer_df.to_csv(out_dir / "peer_bucket_summary.csv", index=False)
by.to_csv(out_dir / "bucket_trend_type_summary.csv", index=False)

print("wrote:", out_dir / "players_summary.csv")
print("wrote:", out_dir / "players_trend_all.csv", f"(n={len(players_trend)})")
print("wrote:", out_dir / "players_trend_incremental.csv", f"(n={len(players_incr)})")
print("wrote:", out_dir / "players_trend_bulk_upload.csv", f"(n={len(players_bulk)})")
print("wrote:", out_dir / "peer_bucket_summary.csv")
print("wrote:", out_dir / "bucket_trend_type_summary.csv")


## 12. Colab / 未挂载数据时的下载说明

导入 cell 找不到数据时会自动 `kagglehub.dataset_download("awmcteam/dx-2026-awmcbot")`。

**Google Colab 先配 Kaggle API：**
1. https://www.kaggle.com/settings → API → Create New Token（`kaggle.json`）
2. 运行下方 cell 上传凭证
3. 再重跑最上面的导入 cell

**Kaggle Notebook：** 右侧 Add Data 挂载 `awmcteam/dx-2026-awmcbot`，一般不必再下载。


In [ ]:
# Google Colab: upload kaggle.json once, then re-run the import cell

# from google.colab import files
# import pathlib, shutil
# uploaded = files.upload()  # choose kaggle.json
# kdir = pathlib.Path.home() / ".kaggle"
# kdir.mkdir(exist_ok=True)
# shutil.move("kaggle.json", kdir / "kaggle.json")
# (kdir / "kaggle.json").chmod(0o600)

# import kagglehub
# from pathlib import Path
# path = kagglehub.dataset_download("awmcteam/dx-2026-awmcbot")
# print("Path:", path)
# print(list(Path(path).rglob("players.jsonl"))[:5])
